In [ ]:
# | default_exp activity

# Activity analysis

> Ground-area-aware metrics of jet activity: per-tile marking density (markings per m²) and per-marking ground area (m²). Pure analysis — lives at top level alongside :mod:`p4tools.classify_by_activity`, not under `production/`.

In [ ]:
# | export
"""activity — ground-area-aware analysis of jet-deposit activity.

Two complementary metrics for asking "is the apparent coverage difference
between Mars Years driven by *more* markings or by *bigger* markings?":

1. :func:`per_tile_marking_density` — fans + blotches per m² of ground area
   per tile. Sensitive to *count* changes.
2. :func:`per_marking_ground_area` — polygon area in m² per individual
   marking. Sensitive to *size* changes.

Both metrics use the per-obsid ``map_scale`` (m/px) from
:func:`p4tools.io.get_metafull` to convert pixel-space polygons into m².
Tile pixel area is fixed at 840×648; ground area scales as ``map_scale²``.

Combined fan + blotch is the default — both are markers of jet activity at
the eruption-event level, separating dilutes the signal. Stratification by
kind is available for diagnostic visibility via the ``kind=`` argument.
"""
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd

from p4tools import io as _io
from p4tools import markings as _mk
from p4tools.coverage import (
    TILE_AREA_PX,
    DEFAULT_CACHE_DIR,
    _row_to_fan,
    _row_to_blotch,
)


def _map_scale_per_obsid(version: str = "v3.1") -> pd.Series:
    """Return ``map_scale`` (m/px) indexed by obsid, for the given catalog version."""
    md = _io.get_meta_data(version)
    if "map_scale" not in md.columns:
        raise RuntimeError(
            f"map_scale column missing from get_meta_data('{version}')"
        )
    if "obsid" in md.columns:
        return md.set_index("obsid")["map_scale"]
    if "OBSERVATION_ID" in md.columns:
        return md.set_index("OBSERVATION_ID")["map_scale"]
    raise RuntimeError("no obsid column found in metadata")


## Tile ground area

In [ ]:
# | export
def tile_ground_area_m2(
    obsids=None,
    *,
    version: str = "v3.1",
) -> pd.Series:
    """Tile ground area in m² per obsid.

    Each P4 tile is 840×648 pixels; ground area = ``840 * 648 * map_scale**2``.
    For v3.1 ``map_scale`` is one of {0.25, 0.5, 1.0} m/px, giving tile
    ground areas of {34 020, 136 080, 544 320} m² respectively.

    Parameters
    ----------
    obsids
        If given, restrict to this iterable of obsid strings; otherwise
        return for all obsids in the catalog version.
    """
    scale = _map_scale_per_obsid(version)
    if obsids is not None:
        scale = scale.reindex(list(obsids))
    return (scale.astype("float64") ** 2 * TILE_AREA_PX).rename("tile_ground_area_m2")


## Per-tile marking density

In [ ]:
# | export
def per_tile_marking_density(
    version: str = "v3.1",
    *,
    kind: str = "all",
) -> pd.DataFrame:
    """Per-tile counts and marking density (markings per m² of ground area).

    Returned columns:
    ``[obsid, tile_id, n_fans, n_blotches, n_markings, map_scale,
       tile_ground_area_m2, density_per_m2]``.

    Parameters
    ----------
    kind : {"all", "fan", "blotch"}
        Which markings to count for ``density_per_m2``. ``"all"`` (default)
        is fan + blotch, the recommended activity proxy. The per-kind counts
        ``n_fans`` / ``n_blotches`` are always present.
    """
    if kind not in ("all", "fan", "blotch"):
        raise ValueError(f"unknown kind: {kind!r}")

    fan_df = _io.get_fan_catalog(version)
    blotch_df = _io.get_blotch_catalog(version)
    n_fans = (
        fan_df.groupby(["obsid", "tile_id"], sort=False)
        .size()
        .rename("n_fans")
        .reset_index()
    )
    n_blotches = (
        blotch_df.groupby(["obsid", "tile_id"], sort=False)
        .size()
        .rename("n_blotches")
        .reset_index()
    )

    tiles = _io.get_tile_coords(version)[["obsid", "tile_id"]].drop_duplicates()
    out = (
        tiles.merge(n_fans, on=["obsid", "tile_id"], how="left")
        .merge(n_blotches, on=["obsid", "tile_id"], how="left")
    )
    out[["n_fans", "n_blotches"]] = (
        out[["n_fans", "n_blotches"]].fillna(0).astype(int)
    )
    out["n_markings"] = out["n_fans"] + out["n_blotches"]

    scale = _map_scale_per_obsid(version)
    out["map_scale"] = out["obsid"].map(scale)
    out["tile_ground_area_m2"] = (out["map_scale"] ** 2) * TILE_AREA_PX

    if kind == "all":
        numerator = out["n_markings"]
    elif kind == "fan":
        numerator = out["n_fans"]
    else:
        numerator = out["n_blotches"]
    out["density_per_m2"] = numerator / out["tile_ground_area_m2"]
    return out


## Per-marking ground area

In [ ]:
# | export
def per_marking_ground_area(
    version: str = "v3.1",
    *,
    kind: str = "both",
    cache: bool = True,
    cache_dir: Path | None = None,
) -> pd.DataFrame:
    """Per-marking polygon area in m².

    Computes Shapely polygons for every fan / blotch using
    ``markings.{Fan,Blotch}.to_shapely`` (HiRISE pixel scope), takes
    ``polygon.area`` in pixel², and converts to m² via per-obsid
    ``map_scale``. Result is cached as parquet next to the coverage cache.

    Parameters
    ----------
    kind : {"both", "fan", "blotch"}
        Which catalog(s) to process. Default ``"both"``.

    Returned columns:
    ``[obsid, marking_id, kind, area_m2]``.
    """
    if kind not in ("both", "fan", "blotch"):
        raise ValueError(f"unknown kind: {kind!r}")
    if cache_dir is None:
        cache_dir = DEFAULT_CACHE_DIR
    cache_dir = Path(cache_dir)
    cache_path = cache_dir / f"PerMarkingArea_{version}_{kind}.parquet"
    if cache and cache_path.exists():
        return pd.read_parquet(cache_path)

    scale = _map_scale_per_obsid(version)

    rows: list[pd.DataFrame] = []
    if kind in ("fan", "both"):
        fan_df = _io.get_fan_catalog(version)
        fans = fan_df.apply(_row_to_fan, axis=1)
        geom = fans.apply(_mk.Fan.to_shapely)
        gdf = gpd.GeoDataFrame(fan_df, geometry=geom)
        gdf["area_px2"] = gdf.area
        gdf["map_scale"] = gdf["obsid"].map(scale)
        gdf["area_m2"] = gdf["area_px2"] * gdf["map_scale"] ** 2
        gdf["kind"] = "fan"
        rows.append(gdf[["obsid", "marking_id", "kind", "area_m2"]])

    if kind in ("blotch", "both"):
        blotch_df = _io.get_blotch_catalog(version)
        blotches = blotch_df.apply(_row_to_blotch, axis=1)
        geom = blotches.apply(_mk.Blotch.to_shapely)
        gdf = gpd.GeoDataFrame(blotch_df, geometry=geom)
        gdf["area_px2"] = gdf.area
        gdf["map_scale"] = gdf["obsid"].map(scale)
        gdf["area_m2"] = gdf["area_px2"] * gdf["map_scale"] ** 2
        gdf["kind"] = "blotch"
        rows.append(gdf[["obsid", "marking_id", "kind", "area_m2"]])

    out = pd.concat(rows, ignore_index=True)
    if cache:
        cache_dir.mkdir(parents=True, exist_ok=True)
        out.to_parquet(cache_path, index=False)
    return out
